# Example 2.2 / 3: A Two-Dimensional Parameter

Companion notebook for **2.2 Lists, Tables and Slices in Data Files** in the AMPL lab Documentation, section "A Two-Dimensional Parameter". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Two-Dimensional Parameter

In Python, a dictionary with tuple keys states the index association explicitly.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;                      # Origins: table rows
set DEST;                      # Destinations: table columns
param cost {ORIG, DEST} >= 0;  # Shipping cost per unit on each route

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIG"] = ["GARY", "CLEV"]
ampl.set["DEST"] = ["FRA", "DET"]
ampl.param["cost"] = {
    ("GARY", "FRA"): 39,
    ("GARY", "DET"): 14,
    ("CLEV", "FRA"): 27,
    ("CLEV", "DET"): 9,
}

In [5]:
# run.py
print(ampl.param["cost"].to_dict())

{('CLEV', 'DET'): 9, ('CLEV', 'FRA'): 27, ('GARY', 'DET'): 14, ('GARY', 'FRA'): 39}


### Pure AMPL: data.dat and model.run

The same values in table form. Each cell is `cost[row_label,column_header]`:

In [6]:
%%writefile data.dat
set ORIG := GARY CLEV;
set DEST := FRA DET;
param cost: FRA DET :=
  GARY 39 14
  CLEV 27  9;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display cost;

cost :=
CLEV DET    9
CLEV FRA   27
GARY DET   14
GARY FRA   39
;

